# 3. Аппроксимация и локальная погрешность

Пусть вместо производной функции мы вычислили наклон секущей. Получилось $1{,}0625$, а точная производная равна $1$. Что произойдёт с ошибкой, если взять вдвое меньший шаг? У одной разностной формулы ошибка уменьшится примерно вдвое, у другой — вчетверо. Посмотрим, откуда берётся это различие.

## 1. Три приближения одной производной

Возьмём функцию

$$
u(x)=x+x^2+x^3.
$$

Её производная в нуле равна $u'(0)=1$. Для секущей можно взять одну из трёх пар точек: $(-h,0)$, $(0,h)$ или $(-h,h)$ по оси $x$. Подстановка $u(0)=0$, $u(h)=h+h^2+h^3$ и $u(-h)=-h+h^2-h^3$ даёт следующие результаты.

| Разность | Наклон секущей | Результат |
|---|---|---|
| Левая | $\displaystyle\frac{u(0)-u(-h)}{h}$ | $1-h+h^2$ |
| Правая | $\displaystyle\frac{u(h)-u(0)}{h}$ | $1+h+h^2$ |
| Центральная | $\displaystyle\frac{u(h)-u(-h)}{2h}$ | $1+h^2$ |

У левой и правой разностей слагаемые $-h$ и $+h$ имеют противоположные знаки. Центральная разность — среднее этих двух наклонов, поэтому слагаемые с $h$ сокращаются. Остаётся ошибка $h^2$.

При $h=1/4$ центральная секущая имеет наклон $1+1/16$, при $h=1/8$ — уже $1+1/64$. Шаг уменьшился вдвое, а ошибка наклона — вчетверо.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as w
    from IPython.display import clear_output, display, Image
    from io import BytesIO
except ImportError:
    w = None

INK, TEAL, RUST = "#15334A", "#007C78", "#A84C26"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#A8B8BF", "axes.labelcolor": INK,
                     "text.color": INK, "font.size": 10, "axes.titlesize": 12,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})


def decimal(x, digits=3):
    return f"{x:.{digits}f}".rstrip("0").rstrip(".").replace(".", ",").replace("-", "−")


def experiment(figure, choices, curves=()):
    defaults = {key: value for key, (_, _, value) in choices.items()}
    if w is None:
        figure(**defaults)
        plt.show()
        return
    selectors = {}
    for key, (label, options, value) in choices.items():
        selectors[key] = w.Dropdown(options=options, value=value, description=label,
                                    style={"description_width": "initial"},
                                    layout=w.Layout(width="300px", max_width="100%"))
    toggles = [w.ToggleButton(value=True, description=label,
                              layout=w.Layout(width="auto")) for label in curves]
    output = w.Output()

    def draw(change=None):
        values = {key: selector.value for key, selector in selectors.items()}
        if toggles:
            values["visible"] = [toggle.value for toggle in toggles]
        fig = figure(**values)
        with BytesIO() as buf:
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", pad_inches=.1)
            picture = Image(data=buf.getvalue(), width=850)
        plt.close(fig)
        with output:
            clear_output(wait=True)
            display(picture)

    for control in [*selectors.values(), *toggles]:
        control.observe(draw, names="value")
    wrap = w.Layout(display="flex", flex_flow="row wrap")
    rows = [w.Box(list(selectors.values()), layout=wrap)]
    if toggles:
        rows.append(w.Box(toggles, layout=wrap))
    display(w.VBox([*rows, output]))
    draw()


def profile(x):
    return x + x**2 + x**3


def tangent_figure(method="central", denominator=4, visible=(True, True, True)):
    h = 1 / denominator
    a, b = {"left": (-h, 0), "right": (0, h), "central": (-h, h)}[method]
    slope = (profile(b) - profile(a)) / (b - a)
    x = np.linspace(-.56, .56, 401)
    fig, ax = plt.subplots(figsize=(8.4, 3.55), constrained_layout=True)
    ax.axhline(0, color="#C5D1D6", lw=.8)
    ax.axvline(0, color="#C5D1D6", lw=.8)
    if visible[0]:
        ax.plot(x, profile(x), color=INK, lw=1.7, label="Функция")
    if visible[1]:
        ax.plot(x, x, "--", color=RUST, lw=1.6, label="Касательная", zorder=3)
    if visible[2]:
        ax.plot(x, slope*(x-a)+profile(a), color=TEAL, lw=2.1, label="Секущая")
        ax.plot([a,b], [profile(a),profile(b)], "o", mfc="white", mec=TEAL, mew=1.6, ms=6)
    ax.set(xlim=(-.56,.56), ylim=(-1.08,1.23), xlabel="Координата x", ylabel="Значение",
           xticks=[-.5,-.25,0,.25,.5], yticks=[-1,-.5,0,.5,1], title="Касательная и секущая вблизи нуля")
    if any(visible):
        ax.legend(loc="upper left", frameon=False, ncol=3, fontsize=9)
    fig.supxlabel("Точная производная  1     ·     Наклон секущей  " + decimal(slope,6)
                  + "     ·     Ошибка по модулю  " + decimal(abs(slope-1),6), fontsize=9)
    return fig


experiment(tangent_figure, {
    "method": ("Формула", [("Левая", "left"), ("Правая", "right"), ("Центральная", "central")], "central"),
    "denominator": ("Шаг h", [(f"1/{n}", n) for n in (2,4,8,16,32)], 4),
}, curves=("Функция", "Касательная", "Секущая и её узлы"))

## 2. Что такое порядок аппроксимации?

Обозначим через $E(h)$ модуль разности между приближённым и точным значениями. Запись $E(h)=O(h^p)$ означает, что при достаточно малом $h$ ошибка ограничена величиной $C h^p$, где $C$ не зависит от шага.

Если главный член ошибки пропорционален $h^p$, то при уменьшении шага вдвое ошибка убывает примерно в $2^p$ раз. Это даёт способ оценить **наблюдаемый порядок** по двум расчётам:

$$
p_{\mathrm{набл}}=\log_2\!\frac{E(h)}{E(h/2)}.
$$

У центральной разности в нашем примере $E(h)=h^2$, поэтому отношение ошибок равно четырём на любой паре шагов. У правой разности $E(h)=h+h^2$. На крупных шагах вклад $h^2$ ещё заметен. По мере измельчения сетки отношение ошибок приближается к двум, а наблюдаемый порядок — к единице.

In [ ]:
def order_figure(coarse=4, visible=(True, True)):
    from matplotlib.ticker import NullLocator
    denominators = 2**np.arange(1,7)
    hs = 1 / denominators
    h = 1 / coarse
    fig, ax = plt.subplots(figsize=(8.4, 3.8), constrained_layout=True)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.axvspan(h/2, h, color="#DFE8EA", alpha=.8)
    ax.grid(axis="y", lw=.7, color="#D7E1E3")
    specs = (("Правая", lambda t: t+t*t, TEAL, "-o"),
             ("Центральная", lambda t: t*t, RUST, "--s"))
    results = []
    for show, (name, error, colour, style) in zip(visible, specs):
        if show:
            ax.plot(hs,error(hs),style,color=colour,ms=4,lw=1.7,label=name)
            chosen = np.array([h/2,h])
            ax.scatter(chosen,error(chosen),s=58,facecolor="white",edgecolor=colour,
                       marker=style[-1],linewidth=1.6,zorder=3)
            ratio = error(h)/error(h/2)
            results.append(f"{name}: E(h)/E(h/2) = {decimal(ratio,2)}, p ≈ {decimal(np.log2(ratio),2)}")
    ax.set(xlim=(1/71,.56),ylim=(1.6e-4,1.28),xlabel="Шаг h",ylabel="Ошибка E(h)",
           title="Ошибка при последовательном делении шага пополам")
    ax.set_xticks(hs, [f"1/{d}" for d in denominators])
    ax.set_yticks([1e-3,1e-2,1e-1,1],["0,001","0,01","0,1","1"])
    ax.yaxis.set_minor_locator(NullLocator())
    if any(visible):
        ax.legend(loc="upper left",frameon=False,ncol=2,fontsize=9)
        fig.supxlabel("      ·      ".join(results), fontsize=9)
    return fig


experiment(order_figure, {
    "coarse": ("Сравниваемые шаги", [(f"1/{n} → 1/{2*n}", n) for n in (2,4,8,16,32)], 4)
}, curves=("Правая разность", "Центральная разность"))

Почему тот же порядок получается и для других гладких функций? Разложение Тейлора около точки $x$ даёт

$$
\frac{u(x+h)-u(x)}{h}
=u'(x)+\frac h2u''(x)+O(h^2).
$$

В правой разности первая поправка к точной производной пропорциональна $h$. В центральной разности при вычитании $u(x+h)-u(x-h)$ все чётные степени $h$ сокращаются. После деления на $2h$ получается

$$
\frac{u(x+h)-u(x-h)}{2h}
=u'(x)+\frac{h^2}{6}u'''(x)+O(h^4).
$$

Отсюда у правой разности **первый порядок аппроксимации**, а у центральной — **второй**. Для отдельной функции результат может оказаться точнее. Например, у квадратичной функции центральная разность вычисляет первую производную точно при любом шаге.

## 3. Как проверить целый шаг разностной схемы?

Пусть $u(x,t)$ — известное точное решение уравнения. Запишем его значения в точках $x_k=kh$ в момент $t_s=s\tau$:

$$
u_k^s=u(x_k,t_s).
$$

Все значения одного временного слоя соберём в вектор $\mathbf u^s$.

Теперь начнём **с точного слоя** $\mathbf u^s$ и сделаем один шаг по разностной схеме. Обозначим действие этого шага через $B_{h,\tau}$. У нас есть два профиля в один и тот же момент времени. Первый, $\mathbf u^{s+1}$, получен из точного решения. Второй, $B_{h,\tau}\mathbf u^s$, вычислен по схеме. Их разность называют **дефектом одного шага**:

$$
\mathbf d^s=\mathbf u^{s+1}-B_{h,\tau}\mathbf u^s.
$$

Достаточно ли для проверки схемы убедиться, что дефект стремится к нулю? Рассмотрим правило, которое просто оставляет профиль на месте. Пусть точное решение — косинус, движущийся вправо с единичной скоростью:

$$
u(x,t)=\cos\bigl(2\pi(x-t)\bigr).
$$

В точке $x=1/4$ начальное значение равно нулю. Через время $\tau$ точное значение равно $\sin(2\pi\tau)$, а наше правило по-прежнему даёт ноль. Дефект $\sin(2\pi\tau)$ стремится к нулю, хотя правило вообще не воспроизводит перенос. Просто за короткое время профиль успевает мало измениться.

Если разделить дефект на длительность шага, ошибка становится видна:

$$
\frac{\sin(2\pi\tau)}{\tau}\longrightarrow 2\pi.
$$

Поэтому для схемы, записанной в форме одного шага, вводят **нормированную невязку аппроксимации**

$$
\boldsymbol\eta^s
=\frac{\mathbf u^{s+1}-B_{h,\tau}\mathbf u^s}{\tau}.
$$

Говорят, что схема **аппроксимирует уравнение**, если для каждого достаточно гладкого точного решения эта невязка стремится к нулю при измельчении сетки. В примерах ниже её величину измеряем по наибольшему модулю среди узлов.

## 4. Невязка для переноса и диффузии

**Перенос.** Уравнение $u_t+u_x=0$ сдвигает профиль вправо с единичной скоростью. Заменим производную по времени разностью вперёд, а производную по пространству — левой разностью. Подстановка точного решения даёт невязку

$$
\eta_k^s=
\frac{u_k^{s+1}-u_k^s}{\tau}
+\frac{u_k^s-u_{k-1}^s}{h}.
$$

Первая дробь приближает $u_t$ с ошибкой $O(\tau)$, вторая — $u_x$ с ошибкой $O(h)$. Поскольку у точного решения $u_t+u_x=0$, остаётся $\eta_k^s=O(\tau+h)$.

**Диффузия.** Уравнение $u_t=u_{xx}$ описывает сглаживание профиля. Оставим ту же разность по времени, а для второй производной по пространству возьмём центральную формулу. Теперь невязка равна

$$
\eta_k^s=
\frac{u_k^{s+1}-u_k^s}{\tau}
-\frac{u_{k+1}^s-2u_k^s+u_{k-1}^s}{h^2}.
$$

Пространственная формула приближает $u_{xx}$ с ошибкой $O(h^2)$. После сокращения $u_t-u_{xx}=0$ получаем $\eta_k^s=O(\tau+h^2)$.

В обоих опытах начальный профиль один и тот же — $\cos(2\pi x)$ на периодическом отрезке длины 1. При переносе точное решение равно $\cos(2\pi(x-t))$, при диффузии — $e^{-4\pi^2t}\cos(2\pi x)$. Это позволяет после одного численного шага непосредственно вычислить и дефект, и невязку.

In [ ]:
def one_step(N, model):
    h = 1/N
    x = np.arange(N)*h
    u = np.cos(2*np.pi*x)
    if model == "advection":
        tau = h/2
        exact = np.cos(2*np.pi*(x-tau))
        step = u - (tau/h)*(u-np.roll(u,1))
    else:
        tau = h*h/4
        exact = np.exp(-4*np.pi**2*tau)*u
        step = u + (tau/h**2)*(np.roll(u,-1)-2*u+np.roll(u,1))
    defect = max(abs(exact-step))
    return tau, defect, defect/tau


def significant(x):
    if x >= .001:
        return f"{x:#.3g}".replace(".", ",")
    power = int(np.floor(np.log10(x)))
    return f"${decimal(x/10**power,2).replace(',', '{,}')}\\cdot 10^{{{power}}}$"


def defect_figure(model="advection", N=32):
    a, b = one_step(N,model), one_step(2*N,model)
    name, time_law = (("Перенос", "τ = h/2") if model == "advection" else ("Диффузия", "τ = h²/4"))
    fig, ax = plt.subplots(figsize=(8.4,2.5), constrained_layout=True)
    ax.set_axis_off()
    ax.set_title(f"{name}. Один шаг на двух сетках   ·   {time_law}", pad=7)
    labels = ("Величина", f"N = {N}", f"N = {2*N}", "Уменьшение")
    names = ("Шаг времени τ", r"Дефект   $\max_k |d_k|$", r"Невязка   $\max_k |\eta_k|$")
    rows = []
    for i, label in enumerate(names):
        fmt = (lambda x: f"1/{round(1/x)}") if i == 0 else significant
        rows.append([label, fmt(a[i]), fmt(b[i]), "×"+decimal(a[i]/b[i],2)])
    table = ax.table(cellText=rows,colLabels=labels,cellLoc="center",colLoc="center",
                     colWidths=[.34,.22,.22,.22],bbox=[0,.04,1,.90])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    for (row,col),cell in table.get_celld().items():
        cell.visible_edges = "B"
        cell.set_edgecolor("#D7E1E3")
        cell.set_linewidth(.8)
        cell.set_text_props(color=INK,weight="medium" if row==0 else "normal")
        if col == 0:
            cell.set_text_props(ha="left")
    return fig


experiment(defect_figure, {
    "model": ("Уравнение", [("Перенос", "advection"), ("Диффузия", "diffusion")], "advection"),
    "N": ("Узлов на первой сетке", [16,32,64,128,256], 32),
})

Для переноса в опыте выбрано $\tau=h/2$. При удвоении числа узлов невязка уменьшается примерно **вдвое**. Сам шаг времени тоже вдвое короче, поэтому дефект $d=\tau\eta$ уменьшается примерно **вчетверо**.

Для диффузии выбрано $\tau=h^2/4$. При удвоении числа узлов и пространственная ошибка $O(h^2)$, и временная ошибка $O(\tau)$ уменьшаются примерно **вчетверо**. Поэтому вчетверо уменьшается невязка. Длительность шага тоже уменьшается вчетверо — отсюда уменьшение дефекта примерно **в шестнадцать раз**.